# Baseline B: Image Data Preprocessing

This notebook prepares the CMMD mammography dataset for image-based deep learning models.

Steps:
1. Load aligned dataset
2. Convert DICOM (.dcm) to image format
3. Resize images
4. Normalize pixel values
5. Assign labels (patient-level)
6. Save processed dataset

In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import pydicom
from tqdm import tqdm

In [6]:
# Update paths based on your system

CMMD_ALIGNED_PATH = r"CMMD_Aligned"
CLINICAL_PATH = r"aligned_clinical_dataset.csv"
OUTPUT_PATH = r"Processed_Images"

os.makedirs(OUTPUT_PATH, exist_ok=True)

In [7]:
df = pd.read_csv(CLINICAL_PATH)

print("Dataset Shape:", df.shape)
print(df.head())

Dataset Shape: (1360, 9)
        ID  target LeftRight   Age  Age_missing  Breast_density_encoded  \
0  D1-0001       0         R  44.0          0.0                     2.0   
1  D1-0002       0         L  40.0          0.0                     2.0   
2  D1-0003       0         L  39.0          0.0                     1.0   
3  D1-0004       0         L  41.0          0.0                     2.0   
4  D1-0005       0         R  42.0          0.0                     3.0   

   BI_RADS  Mass_present  Calc_present  
0      2.0           0.0           1.0  
1      2.0           0.0           1.0  
2      3.0           0.0           1.0  
3      2.0           0.0           1.0  
4      2.0           0.0           1.0  


In [8]:
# Map Patient ID → Target
label_dict = dict(zip(df["ID"], df["target"]))

print("Total labels:", len(label_dict))

Total labels: 1360


Each patient ID is mapped to a binary target:
- 0 → Benign
- 1 → Malignant

This ensures patient-level classification consistency.

In [9]:
def dicom_to_image(dcm_path):
    dcm = pydicom.dcmread(dcm_path)
    img = dcm.pixel_array

    # Normalize to 0–255
    img = img - np.min(img)
    img = img / np.max(img)
    img = (img * 255).astype(np.uint8)

    return img

In [10]:
IMG_SIZE = 224

data = []

for patient_id in tqdm(os.listdir(CMMD_ALIGNED_PATH)):
    patient_path = os.path.join(CMMD_ALIGNED_PATH, patient_id)

    if patient_id not in label_dict:
        continue

    label = label_dict[patient_id]

    for root, _, files in os.walk(patient_path):
        for file in files:
            if file.endswith(".dcm"):
                dcm_path = os.path.join(root, file)

                try:
                    img = dicom_to_image(dcm_path)

                    # Resize
                    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

                    # Save image
                    save_name = f"{patient_id}_{file.replace('.dcm', '.png')}"
                    save_path = os.path.join(OUTPUT_PATH, save_name)

                    cv2.imwrite(save_path, img)

                    # Store metadata
                    data.append([save_name, label])

                except Exception as e:
                    print("Error:", dcm_path)

  0%|          | 0/1360 [00:00<?, ?it/s]

100%|██████████| 1360/1360 [05:52<00:00,  3.86it/s]


In [11]:
processed_df = pd.DataFrame(data, columns=["image_name", "label"])

processed_df.to_csv("processed_labels.csv", index=False)

print("Saved processed_labels.csv")
print(processed_df.head())

Saved processed_labels.csv
        image_name  label
0  D1-0001_1-1.png      0
1  D1-0001_1-2.png      0
2  D1-0002_1-1.png      0
3  D1-0002_1-2.png      0
4  D1-0003_1-1.png      0


## Output

- Folder: Processed_Images/
- CSV: processed_labels.csv

Each row contains:
- image_name
- label

This dataset is now ready for deep learning training.